In [28]:
import warnings
warnings.filterwarnings('ignore')

In [31]:
import os
import gc

import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tqdm as notebook_tqdm

from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import RocCurveDisplay, roc_curve, auc, classification_report, confusion_matrix, accuracy_score, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchinfo import summary

from utils import reduce_mem_usage, plot_confusion_matrix

In [3]:
# задаем параметры
BATCH_SIZE = 1000
N_EPOCHS = 30
VAL_STEPS = 2
L_RATE = 0.0005

In [5]:
data_train = pd.read_parquet(r'../prep_data/data_train.pq')
data_test = pd.read_parquet(r'../prep_data/data_test.pq')

In [6]:
data_train.shape, data_test.shape

((2400000, 388), (600000, 388))

In [7]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(data_train['flag']),
    y=data_train['flag']
)
class_weights

array([ 0.51839302, 14.09211546])

In [8]:
X_train, y_train = data_train.drop(columns=['id', 'flag']), data_train[['flag']]
X_test, y_test = data_test.drop(columns=['id', 'flag']), data_test[['flag']]

In [9]:
del data_train, data_test
gc.collect()

902

In [10]:
class MyDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X.values, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32)
    
    def __len__(self):
        return self.X.shape[0]
        
    def __getitem__(self, index):
        return self.X[index], self.y[index]

In [11]:
train_dataset = MyDataset(X_train, y_train)
test_dataset = MyDataset(X_test, y_test)

In [12]:
train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,          
    num_workers=4,
    drop_last=True
)

In [13]:
test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    drop_last=False
)

In [14]:
class ClassificationNet(nn.Module):
    def __init__(self, in_feature):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Linear(in_feature, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5)
        )

        self.layer2 = nn.Sequential(
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.5)
        )

        self.layer3 = nn.Sequential(
            nn.Linear(64, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
        
    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = self.layer3(out)
        return out

In [15]:
model = ClassificationNet(in_feature=len(X_train.columns))
summary(model, input_size=(1, 386))

Layer (type:depth-idx)                   Output Shape              Param #
ClassificationNet                        [1, 1]                    --
├─Sequential: 1-1                        [1, 256]                  --
│    └─Linear: 2-1                       [1, 256]                  99,072
│    └─BatchNorm1d: 2-2                  [1, 256]                  512
│    └─ReLU: 2-3                         [1, 256]                  --
│    └─Dropout: 2-4                      [1, 256]                  --
├─Sequential: 1-2                        [1, 64]                   --
│    └─Linear: 2-5                       [1, 64]                   16,448
│    └─BatchNorm1d: 2-6                  [1, 64]                   128
│    └─ReLU: 2-7                         [1, 64]                   --
│    └─Dropout: 2-8                      [1, 64]                   --
├─Sequential: 1-3                        [1, 1]                    --
│    └─Linear: 2-9                       [1, 16]                   1,040
│ 

In [16]:
# объявляем оптимизатор
optimizer = torch.optim.Adam(model.parameters(), lr=L_RATE)

In [17]:
pos_weight = torch.tensor([class_weights[1] / class_weights[0]])
pos_weight

tensor([27.1842], dtype=torch.float64)

In [18]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [19]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
criterion = criterion.to(device)

In [20]:
def binary_accuracy(preds, y):
    rounded_preds = torch.round(torch.sigmoid(preds))
    correct = (rounded_preds == y).float()
    acc = correct.sum() / len(correct)
    return acc

In [21]:
def train(model, dataloader, optimizer, criterion):

    model.train()

    epoch_loss = 0
    all_logits = []
    all_labels = []

    for xb, yb in dataloader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        epoch_loss += loss.item()
        all_logits += logits.tolist()
        all_labels += yb.tolist()
        loss.backward()
        optimizer.step()

    all_logits = torch.tensor(all_logits)
    all_labels = torch.tensor(all_labels)

    y_pred_binary = torch.round(torch.sigmoid(all_logits)).cpu().numpy()
    y_true = all_labels.cpu().numpy()

    accuracy = accuracy_score(y_true, y_pred_binary)
    roc_auc = roc_auc_score(y_true, all_logits.cpu().numpy())
   

    return epoch_loss / len(dataloader), accuracy, roc_auc

In [22]:
def validate(model, dataloader, criterion):

    model.eval()

    val_epoch_loss = 0
    val_all_logits = []
    val_all_labels = []

    for xb, yb in dataloader:
        xb, yb = xb.to(device), yb.to(device)
        with torch.no_grad():
            logits = model(xb)
            loss = criterion(logits, yb)
        val_epoch_loss += loss.item()
        val_all_logits += logits.tolist()
        val_all_labels += yb.tolist()
    
    val_all_logits = torch.tensor(val_all_logits)
    val_all_labels = torch.tensor(val_all_labels)

    y_pred_binary = torch.round(torch.sigmoid(val_all_logits)).cpu().numpy()
    y_true = val_all_labels.cpu().numpy()

    val_accuracy = accuracy_score(y_true, y_pred_binary)
    val_roc_auc = roc_auc_score(y_true, val_all_logits.cpu().numpy())

    return val_epoch_loss/len(dataloader), val_accuracy, val_roc_auc

In [23]:
# объявляем списки метрик
epoch_losses = {}
epoch_accuracies = {}
epoch_roc_aucs = {}
val_losses = {}
val_accuracies = {}
val_roc_aucs = {}

In [24]:
%%time
for epoch in notebook_tqdm.tqdm(range(N_EPOCHS)):
    
    if (epoch + 1)%VAL_STEPS == 0:
        epoch_loss, epoch_accuracy, epoch_roc_auc = train(model, train_dataloader, optimizer, criterion)
        val_loss, val_accuracy, val_roc_auc = validate(model, test_dataloader, criterion)
        epoch_losses[epoch + 1] = epoch_loss
        epoch_accuracies[epoch + 1] = epoch_accuracy
        epoch_roc_aucs[epoch + 1] = epoch_roc_auc
        val_losses[epoch + 1] = val_loss
        val_accuracies[epoch + 1] = val_accuracy
        val_roc_aucs[epoch + 1] = val_roc_auc
        print(f'Epoch {epoch+1:02}, Epoch loss = {epoch_loss:.2f}, Epoch accuracy = {epoch_accuracy:.3f}, Epoch ROC AUC = {epoch_roc_auc:.3f}', 
              f'Val loss = {val_loss:.2f}, Val accuracy = {val_accuracy:.3f}, Val ROC AUC = {val_roc_auc:.3f}')
    else:
        epoch_loss, epoch_accuracy, epoch_roc_auc = train(model, train_dataloader, optimizer, criterion)
        epoch_losses[epoch + 1] = epoch_loss
        epoch_accuracies[epoch + 1] = epoch_accuracy
        epoch_roc_aucs[epoch + 1] = epoch_roc_auc
        print(f'Epoch {epoch+1:02}, Epoch loss = {epoch_loss:.2f}, Epoch accuracy = {epoch_accuracy:.3f}, Epoch ROC AUC = {epoch_roc_auc:.3f}')

  3%|████▊                                                                                                                                            | 1/30 [00:36<17:37, 36.48s/it]

Epoch 01, Epoch loss = 1.19, Epoch accuracy = 0.613, Epoch ROC AUC = 0.720


  7%|█████████▋                                                                                                                                       | 2/30 [01:15<17:37, 37.75s/it]

Epoch 02, Epoch loss = 1.16, Epoch accuracy = 0.630, Epoch ROC AUC = 0.738 Val loss = 1.14, Val accuracy = 0.616, Val ROC AUC = 0.752


 10%|██████████████▌                                                                                                                                  | 3/30 [01:50<16:29, 36.64s/it]

...

Epoch 28, Epoch loss = 1.11, Epoch accuracy = 0.661, Epoch ROC AUC = 0.767 Val loss = 1.13, Val accuracy = 0.651, Val ROC AUC = 0.762


 97%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏    | 29/30 [18:12<00:37, 37.40s/it]

Epoch 29, Epoch loss = 1.11, Epoch accuracy = 0.659, Epoch ROC AUC = 0.768


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30/30 [18:51<00:00, 37.70s/it]

Epoch 30, Epoch loss = 1.11, Epoch accuracy = 0.660, Epoch ROC AUC = 0.768 Val loss = 1.13, Val accuracy = 0.653, Val ROC AUC = 0.762
CPU times: user 1h 23min 25s, sys: 2min 29s, total: 1h 25min 55s
Wall time: 18min 51s


In [33]:
torch.save(model.state_dict(), 'nn.pth')